# QuranicTools — Full Functionality Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/language-ml/hadith-quranic_nlp/blob/main/notebooks/quranic_nlp_demo.ipynb)

This notebook demonstrates all available features of the `quranic-nlp` library.

**Contents:**
1. [Installation & Data Download](#1-installation--data-download)
2. [Pipeline Setup](#2-pipeline-setup)
3. [Input Formats](#3-input-formats)
4. [Verse Information](#4-verse-information)
5. [Word-level Information](#5-word-level-information)
6. [JSON Output](#6-json-output)
7. [Multiple Matches](#7-multiple-matches)
8. [Translations](#8-translations)
9. [Hadiths](#9-hadiths)
10. [Visualization](#10-visualization)

## 1. Installation & Data Download

In [ ]:
!pip install quranic-nlp

In [ ]:
from quranic_nlp.data_requirements import download_data
download_data()

## 2. Pipeline Setup

Available pipeline components:
- `dep` — Dependency parsing
- `pos` — Part-of-speech tagging
- `root` — Root extraction
- `lem` — Lemmatization

In [ ]:
from quranic_nlp import language, utils

# Full pipeline with Persian (Farsi) translation, translator index 1
translation_lang = 'fa#1'
pips = 'dep,pos,root,lem'

nlp = language.Pipeline(pips, translation_lang)
print('Pipeline ready.')

## 3. Input Formats

Three ways to reference a verse:

In [ ]:
# Format 1: surah_number#ayah_number
doc = nlp('1#1')
print('Format 1 (numeric):', doc)

In [ ]:
# Format 2: surah_name#ayah_number (requires internet)
doc = nlp('حمد#1')
print('Format 2 (surah name):', doc)

In [ ]:
# Format 3: free Arabic text search (requires internet)
doc = nlp('رب العالمین')
print('Format 3 (free text search):', doc)

## 4. Verse Information

In [ ]:
# Surah Al-Fatiha, Verse 1
doc1 = nlp('1#1')

print('Text (with diacritics):', doc1)
print('Full text:', doc1._.text)
print('Surah name:', doc1._.surah)
print('Ayah number:', doc1._.ayah)
print('Revelation order:', doc1._.revelation_order)
print('Translation:', doc1._.translations)
print('Similar verses:', doc1._.sim_ayahs[:5], '...')

In [ ]:
# Surah Aal-i-Imran, Verse 200
doc2 = nlp('3#200')

print('Text:', doc2)
print('Full text:', doc2._.text)
print('Surah name:', doc2._.surah)
print('Ayah number:', doc2._.ayah)
print('Revelation order:', doc2._.revelation_order)
print('Translation:', doc2._.translations)

## 5. Word-level Information

In [ ]:
# Word-level analysis for Surah Al-Fatiha, Verse 1 — third word
word = doc1[2]

print('Word:', word)
print('POS tag:', word.pos_)
print('POS tag (Arabic):', utils.POS_UNI_FA[word.pos_])
print('Dependency:', word.dep_)
print('Dependency arc:', word._.dep_arc)
print('Head:', word.head)
print('Lemma:', word.lemma_)
print('Root:', word._.root)

In [ ]:
# All words in Surah Al-Fatiha, Verse 1
print(f'{'Word':<25} {'POS':<8} {'Lemma':<20} {'Root':<15} {'Dep'}')
print('-' * 80)
for token in doc1:
    print(f'{str(token):<25} {token.pos_:<8} {token.lemma_:<20} {str(token._.root):<15} {token.dep_}')

## 6. JSON Output

In [ ]:
import json

result = language.to_json(pips, doc1)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 7. Multiple Matches

When searching by free text, a query may match multiple verses.
Use `language.search_all(nlp, text)` to get a list of docs for all matches:

In [ ]:
# Get all verses matching a free text query (top 5)
docs = language.search_all(nlp, 'رب العالمین', max_results=5)

print(f'Found {len(docs)} matching verses:\n')
for doc in docs:
    print(f'  Surah {doc._.surah} ({doc._.ayah}): {doc._.text}')

## 7. Translations

The library supports 43 languages. See all available translators:

In [ ]:
utils.print_all_translations()

In [ ]:
# English translation (Yusuf Ali)
nlp_en = language.Pipeline(pips, 'en#16')
doc_en = nlp_en('1#1')
print('English (Yusuf Ali):', doc_en._.translations)

In [ ]:
# Persian translation (Makarem Shirazi)
nlp_fa = language.Pipeline(pips, 'fa#8')
doc_fa = nlp_fa('1#1')
print('Persian (Makarem Shirazi):', doc_fa._.translations)

## 8. Hadiths

Fetch related hadiths for a verse (requires internet):

In [ ]:
hadiths = doc1._.hadiths
if hadiths:
    print(f'Found {len(hadiths)} hadith(s):\n')
    for i, h in enumerate(hadiths[:2], 1):
        print(f'--- Hadith {i} ---')
        print(h)
        print()
else:
    print('No hadiths found or API unavailable.')

## 9. Visualization

Render the dependency parse tree using spaCy's displacy:

In [ ]:
from spacy import displacy

options = {
    'compact': True,
    'bg': '#09a3d5',
    'color': 'white',
    'font': 'Arial'
}

displacy.render(doc1, style='dep', options=options, jupyter=True)